In [6]:
import torch
import torch.nn as nn

token_ids = torch.tensor([0, 1, 1, 2, 3], dtype = torch.long)
target_ids = torch.tensor([1,1,2,3], dtype= torch.long)
number_of_tokens = len(token_ids)

class ToyAttentionModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.E = nn.Parameter(
            torch.tensor([[1.0, 0.0, 1.0, 0.0],
                          [0.0, 1.0, 0.0, 1.0],
                          [1.0, 1.0, 0.0, 0.0],
                          [0.0, 0.0, 1.0, 1.0]])
        )
        self.W_Q = nn.Parameter(torch.tensor([[1.0,0.0],
                   [0.0,1.0],
                   [1.0,0.0],
                   [0.0,1.0]]))

        self.W_K = nn.Parameter(torch.tensor([[1.0,0.0],
                   [0.0,1.0],
                   [0.0,1.0],
                   [1.0,0.0]]))
        self.W_V = nn.Parameter(torch.tensor([[1.0,0.0,0.0],
                    [0.0,1.0,0.0],
                    [0.0,0.0,1.0],
                    [1.0,1.0,0.0]])) 
        self.W_vocab = nn.Parameter( torch.tensor([[1.0,0.0,-1.0,0.0],
                         [0.0,1.0,1.0,-1.0],
                         [-1.0,0.0,0.0,1.0]]))
    def forward(self, token_ids):        
        
        

        X = self.E[token_ids]
        Q = X @ self.W_Q
        K = X @ self.W_K
        V = X @ self.W_V
        d_k = Q.shape[1]

        S = (Q @ K.T)/(d_k)**0.5

        M = torch.full((number_of_tokens, number_of_tokens), float("-inf"), dtype = X.dtype, device = X.device)
        M = torch.triu(M, diagonal = 1)

        S_masked = S + M
        A = torch.softmax(S_masked, dim = 1)
        O = A @ V
        O_train = O[:-1]
        Z = O_train @ self.W_vocab
        return Z

model = ToyAttentionModel()
logits = model(token_ids)

print(logits.shape)


print(model.E)
print(model.E.requires_grad)

for name, parameter in model.named_parameters():
    print(name, parameter.shape)



torch.Size([4, 4])
Parameter containing:
tensor([[1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [1., 1., 0., 0.],
        [0., 0., 1., 1.]], requires_grad=True)
True
E torch.Size([4, 4])
W_Q torch.Size([4, 2])
W_K torch.Size([4, 2])
W_V torch.Size([4, 3])
W_vocab torch.Size([3, 4])
